In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn

In [3]:
import torch.optim as optim

In [4]:
from torchvision import datasets

In [5]:
from torchvision import transforms

In [57]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

In [7]:
import os

print(os.listdir("/content"))

['.config', 'drive', 'sample_data']


In [11]:
import zipfile

with zipfile.ZipFile('/content/cifar10.zip', 'r') as zip_ref:
    print(zip_ref.namelist()[:20])

['cifar10/labels.txt', 'cifar10/test/airplane/1001_airplane.png', 'cifar10/test/airplane/1010_airplane.png', 'cifar10/test/airplane/1018_airplane.png', 'cifar10/test/airplane/1022_airplane.png', 'cifar10/test/airplane/1023_airplane.png', 'cifar10/test/airplane/1026_airplane.png', 'cifar10/test/airplane/1027_airplane.png', 'cifar10/test/airplane/1036_airplane.png', 'cifar10/test/airplane/1052_airplane.png', 'cifar10/test/airplane/1067_airplane.png', 'cifar10/test/airplane/1072_airplane.png', 'cifar10/test/airplane/1076_airplane.png', 'cifar10/test/airplane/1077_airplane.png', 'cifar10/test/airplane/1078_airplane.png', 'cifar10/test/airplane/1090_airplane.png', 'cifar10/test/airplane/1093_airplane.png', 'cifar10/test/airplane/10_airplane.png', 'cifar10/test/airplane/1103_airplane.png', 'cifar10/test/airplane/1109_airplane.png']


In [12]:
import zipfile

with zipfile.ZipFile('/content/cifar10.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
train_dataset = datasets.ImageFolder(
    root = "/content/cifar10/train",
    transform = transform
)

In [15]:
test_dataset = datasets.ImageFolder(
    root = "/content/cifar10/test",
    transform = transform
)

In [33]:
print(len(train_dataset))
print(len(test_dataset))

50000
10000


In [34]:
from torch.utils.data import DataLoader

In [35]:
train_loader = DataLoader(
    dataset = train_dataset,
    batch_size = 64,
    shuffle = True
)

In [36]:
test_loader = DataLoader(
    dataset = test_dataset,
    batch_size = 64,
    shuffle = False
)

In [37]:
nn.Conv2d(
    in_channels = 3,
    out_channels = 32,
    kernel_size = 3
)

Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))

In [38]:
nn.ReLU()

ReLU()

In [39]:
nn.MaxPool2d(2,2)

MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)

In [61]:
class CNN(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv1 = nn.Conv2d(
        in_channels = 3,
        out_channels = 32,
        kernel_size = 3
    )

    self.conv2 = nn.Conv2d(
        in_channels = 32,
        out_channels = 64,
        kernel_size = 3
    )

    self.conv3 = nn.Conv2d(
        in_channels = 64,
        out_channels = 128,
        kernel_size = 3
    )

    self.pool = nn.MaxPool2d(2,2)

    self.fc1 = nn.Linear(2048,128)
    self.fc2 = nn.Linear(128,10)

  def forward(self,x):

    x = self.conv1(x)

    x = torch.relu(x)

    x = self.pool(x)

    x = self.conv2(x)

    x = torch.relu(x)

    x = self.pool(x)

    x = self.conv3(x)

    x = torch.relu(x)

    x = x.view(
        x.size(0),
        2048
    )

    x = self.fc1(x)

    x = torch.relu(x)

    x = self.fc2(x)

    return x

In [72]:
model = CNN()

In [63]:
for images, labels in train_loader:
  outputs = model(images)
  print(outputs.shape)
  break

torch.Size([64, 10])


In [73]:
criterion = nn.CrossEntropyLoss()

In [74]:
optimizer = optim.Adam(
    model.parameters(),
    lr = 0.001
)

In [77]:
epochs = 15
for epoch in range(epochs):

    running_loss = 0

    for images, labels in train_loader:

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}, "
        f"Loss: {running_loss/len(train_loader):.4f}"
    )

Epoch 1, Loss: 1.2137
Epoch 2, Loss: 1.0747
Epoch 3, Loss: 0.9681
Epoch 4, Loss: 0.8841
Epoch 5, Loss: 0.8145
Epoch 6, Loss: 0.7527
Epoch 7, Loss: 0.7047
Epoch 8, Loss: 0.6663
Epoch 9, Loss: 0.6196
Epoch 10, Loss: 0.5842
Epoch 11, Loss: 0.5524
Epoch 12, Loss: 0.5177
Epoch 13, Loss: 0.4894
Epoch 14, Loss: 0.4573
Epoch 15, Loss: 0.4246


In [78]:
correct = 0
total = 0

for images,labels in test_loader:
  outputs = model(images)

  _, predicted = torch.max(outputs, dim=1)

  total += labels.size(0)

  correct += (predicted == labels).sum().item()

In [79]:
accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 71.08%


In [80]:
torch.save(
    model.state_dict(),
    "cifar10_cnn.pth"
)